# KG1 v73 ROBUSTO FINAL - Funciona em H100 80GB E A100 40GB/80GB

## Hardware suportado:
- **H100 80GB** (recomendado): Unsloth BF16 ~63GB, treino 3-4h
- **A100 80GB**: Unsloth BF16 ~63GB, treino 4-5h
- **A100 40GB**: Unsloth com NF4 forcado ~18GB, treino 5-7h
- Qualquer GPU >=24GB VRAM + sm_70+

## Strategy:
1. Detecta VRAM
2. Cell 4 tenta Unsloth (mais rapido)
3. Fallback automatico transformers + NF4 direct se Unsloth falhar
4. Detectar path usado (USED_UNSLOTH flag)
5. Cell 5 aplica LoRA compativel com path escolhido

## Config kienngx:
- r=32, alpha=32, dropout=0.05
- target_modules 8 modules explicit
- lr=1e-4, cosine + warmup 0.1
- bs=1 ga=4, 2 epochs, max_length=2048
- Dataset train.csv sample(1200, seed=42)

## Score target: 0.85 +/- 0.02 (kienngx replica)

In [ ]:
# Cell 1: Install deps
%%capture
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps 'trl>=0.16' 'peft>=0.18.1' accelerate bitsandbytes
!pip install -q 'transformers>=4.55' datasets hf_transfer


In [ ]:
# Cell 2: GPU check relaxado + disable broken unsloth patch
import os, subprocess, sys
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["UNSLOTH_IS_PRESENT"] = "0"

# Disable broken unsloth patch
try:
    import unsloth_zoo
    uzoo_path = os.path.dirname(unsloth_zoo.__file__)
    misc_path = f"{uzoo_path}/temporary_patches/misc.py"
    if os.path.exists(misc_path):
        with open(misc_path, "r") as f:
            content = f.read()
        patch_line = "TEMPORARY_PATCHES.append(patch_merge_quantization_configs)"
        if patch_line in content and f"# DISABLED: {patch_line}" not in content:
            with open(misc_path, "w") as f:
                f.write(content.replace(patch_line, f"# DISABLED: {patch_line}"))
            print(f"[OK] Disabled broken patch")
            for mod_name in list(sys.modules):
                if "unsloth" in mod_name.lower(): del sys.modules[mod_name]
except ImportError: pass

# GPU diagnostic
r = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv",
                   shell=True, capture_output=True, text=True)
print(r.stdout)

import torch
assert torch.cuda.is_available(), "CUDA indisponivel"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cc = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu} | VRAM: {vram:.1f}GB | sm_{cc[0]}{cc[1]}")
print(f"Torch: {torch.__version__}")

assert vram >= 24, f"VRAM {vram:.1f}GB < 24GB"
assert cc[0] >= 7, f"sm_{cc[0]}{cc[1]} incompativel"

# Detectar strategy
if vram >= 70:
    STRATEGY = "unsloth_bf16"
    print(f"[INFO] Strategy: Unsloth BF16 (comprovado em H100 80GB)")
elif vram >= 24:
    STRATEGY = "nf4_forced"
    print(f"[INFO] Strategy: NF4 forcado ({vram:.1f}GB < 70GB, BF16 nao cabe)")

from IPython.display import display, Javascript
display(Javascript("function ClickConnect(){document.querySelector('colab-connect-button').click()};setInterval(ClickConnect, 60000)"))
print(f"[OK] Env ready | STRATEGY={STRATEGY}")


In [ ]:
# Cell 3: Drive + HF + Kaggle secrets
import os
from google.colab import drive, userdata

drive.mount("/content/drive")

try:
    HF_TOKEN = userdata.get("HF_KEY")
except Exception:
    try: HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception: HF_TOKEN = ""
assert HF_TOKEN.startswith("hf_"), "Configure HF_KEY nos Secrets"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"[OK] HF_TOKEN: {HF_TOKEN[:10]}...")

try:
    KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
    KAGGLE_KEY = userdata.get("KAGGLE_KEY")
    os.makedirs("/root/.kaggle", exist_ok=True)
    import json as _json
    with open("/root/.kaggle/kaggle.json", "w") as f:
        _json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print(f"[OK] Kaggle: {KAGGLE_USERNAME}")
except Exception as e:
    print(f"WARN: Kaggle creds: {e}")

CKPT_DIR = "/content/drive/MyDrive/kg1_v73_robusto"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"[OK] CKPT_DIR: {CKPT_DIR}")


In [ ]:
# Cell 4: Load model HYBRID (Unsloth OR transformers NF4 fallback)
import sys, torch, gc, shutil, glob
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

# Clear stale caches
for mod_name in list(sys.modules):
    if "nemotron" in mod_name.lower() or "Nemotron" in mod_name or "unsloth" in mod_name.lower():
        del sys.modules[mod_name]

modules_base = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules")
if os.path.exists(modules_base):
    for item in os.listdir(modules_base):
        if "nemotron" in item.lower() or "Nemotron" in item:
            shutil.rmtree(f"{modules_base}/{item}", ignore_errors=True)

MAX_SEQ = 2048
MODEL_ID = "unsloth/Nemotron-3-Nano-30B-A3B"

USED_UNSLOTH = False
model = None
tok = None

# TENTATIVA 1: Unsloth (H100 ou A100 com Unsloth auto-NF4)
try:
    import unsloth
    from unsloth import FastLanguageModel
    print(f"Trying Unsloth path (strategy={STRATEGY})...")
    model, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ,
        load_in_4bit=True,
        full_finetuning=False,
        token=HF_TOKEN,
        dtype=torch.bfloat16,
    )
    USED_UNSLOTH = True
    print(f"[OK] Loaded via Unsloth")
except Exception as e:
    print(f"[WARN] Unsloth failed ({type(e).__name__}): {str(e)[:200]}")
    print("[INFO] Fallback para transformers + NF4 direct...")
    if model is not None: del model
    if tok is not None: del tok
    gc.collect(); torch.cuda.empty_cache()

    # TENTATIVA 2: transformers + BitsAndBytesConfig NF4 direct
    # Precisa patch rmsnorm se mamba-ssm nao disponivel
    try:
        import mamba_ssm
        MAMBA_OK = True
    except ImportError:
        MAMBA_OK = False
        print("[INFO] mamba-ssm nao disponivel, aplicando patch gated rmsnorm")
        # Patch modeling_nemotron_h.py
        from huggingface_hub import snapshot_download
        hub_dir = snapshot_download(
            MODEL_ID,
            allow_patterns=["*.py", "config.json", "tokenizer*", "special_tokens*", "chat_template*"],
            token=HF_TOKEN, force_download=True,
        )
        def patch_file(model_py):
            if not os.path.exists(model_py): return False
            with open(model_py) as f: content = f.read()
            if "# patched_gated" in content: return True
            OLD = ('try:\n    #from mamba_ssm.ops.triton.layernorm_gated import RMSNorm as RMSNormGated\n'
                   '    from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn\nexcept ImportError:\n'
                   '    raise ImportError("mamba-ssm is required by the Mamba model but cannot be imported")')
            NEW = ('# patched_gated pure-PyTorch\ntry:\n    from mamba_ssm.ops.triton.layernorm_gated import rmsnorm_fn\n'
                   'except ImportError:\n    import torch.nn.functional as _F\n'
                   '    def rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-6, group_size=None, '
                   'norm_before_gate=False, is_rms_norm=True, **kwargs):\n'
                   '        dtype = x.dtype\n'
                   '        if z is not None: x = x * _F.silu(z)\n'
                   '        if group_size is not None and group_size > 0 and x.shape[-1] != group_size:\n'
                   '            orig_shape = x.shape\n'
                   '            x = x.reshape(*orig_shape[:-1], -1, group_size)\n'
                   '            variance = x.pow(2).mean(-1, keepdim=True)\n'
                   '            x = x * torch.rsqrt(variance + eps)\n'
                   '            x = x.reshape(orig_shape)\n'
                   '        else:\n'
                   '            variance = x.pow(2).mean(-1, keepdim=True)\n'
                   '            x = x * torch.rsqrt(variance + eps)\n'
                   '        x = x * weight\n'
                   '        if bias is not None: x = x + bias\n'
                   '        return x.to(dtype)')
            if OLD not in content: return False
            content = content.replace(OLD, NEW)
            if "is_fast_path_available = all(" in content:
                content = content.replace("is_fast_path_available = all(", "is_fast_path_available = False\nis_fast_path_available_orig = all(")
            if "import torch" not in content[:500]:
                content = "import torch\n" + content
            with open(model_py, "w") as f: f.write(content)
            return True
        patch_file(f"{hub_dir}/modeling_nemotron_h.py")
        try:
            from transformers.dynamic_module_utils import get_class_from_dynamic_module
            try: _ = get_class_from_dynamic_module("modeling_nemotron_h.NemotronHForCausalLM", MODEL_ID, token=HF_TOKEN)
            except Exception: pass
        except Exception: pass
        for p in glob.glob(os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/**/modeling_nemotron_h.py"), recursive=True):
            patch_file(p)
        for mod_name in list(sys.modules):
            if "nemotron" in mod_name.lower(): del sys.modules[mod_name]

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map={"": 0},
        token=HF_TOKEN,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    USED_UNSLOTH = False
    print(f"[OK] Loaded via transformers NF4 direct")

if tok.pad_token is None: tok.pad_token = tok.eos_token

vram_used = torch.cuda.memory_allocated() / 1e9
print(f"\n[OK] Model loaded.")
print(f"  USED_UNSLOTH: {USED_UNSLOTH}")
print(f"  Class: {type(model).__name__}")
print(f"  MAX_SEQ: {MAX_SEQ}")
print(f"  GPU mem: {vram_used:.1f}GB")


In [ ]:
# Cell 5: LoRA kienngx - compatível com Unsloth OR transformers
TARGET_MODULES = ["in_proj", "out_proj", "q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj"]

LORA_KWARGS = dict(
    r=32, lora_alpha=32, lora_dropout=0.05,
    bias="none",
    target_modules=TARGET_MODULES,
)

if USED_UNSLOTH:
    from unsloth import FastLanguageModel
    model = FastLanguageModel.get_peft_model(
        model,
        target_parameters=[],   # desabilita MoE
        use_rslora=False, use_dora=False,
        use_gradient_checkpointing="unsloth",
        random_state=42,
        **LORA_KWARGS,
    )
    print("[OK] LoRA via Unsloth (sem MoE)")
else:
    from peft import LoraConfig, get_peft_model, TaskType
    cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        **LORA_KWARGS,
    )
    model = get_peft_model(model, cfg)
    model.enable_input_require_grads()
    print("[OK] LoRA via PEFT direct")

model.print_trainable_parameters()

import torch
vram = torch.cuda.memory_allocated() / 1e9
print(f"GPU mem apos LoRA: {vram:.1f}GB")


In [ ]:
# Cell 6: Dataset train.csv OFICIAL Kaggle (kienngx 1200 seed=42)
import os, shutil
import pandas as pd
from datasets import Dataset

TRAIN_CSV = "/content/drive/MyDrive/kg1_train.csv"
if not os.path.exists(TRAIN_CSV):
    print("Baixando train.csv via Kaggle API...")
    assert os.path.exists("/root/.kaggle/kaggle.json"), "Configure KAGGLE creds"
    os.system("kaggle competitions download -c nvidia-nemotron-model-reasoning-challenge -f train.csv -p /content/ 2>&1")
    os.system("unzip -o /content/train.csv.zip -d /content/ 2>/dev/null || true")
    if os.path.exists("/content/train.csv"):
        TRAIN_CSV = "/content/train.csv"
        shutil.copy(TRAIN_CSV, "/content/drive/MyDrive/kg1_train.csv")

df_full = pd.read_csv(TRAIN_CSV)
print(f"Full: {len(df_full)} rows | cols: {list(df_full.columns)}")

df = df_full.sample(n=1200, random_state=42).reset_index(drop=True)
print(f"Subsampled kienngx: {len(df)}")

PROMPT_COL = "prompt" if "prompt" in df.columns else "problem"
ANSWER_COL = "answer" if "answer" in df.columns else "solution"
PROMPT_SUFFIX = chr(10) + "Put your final answer inside \\boxed{}."

def format_kienngx(row):
    user_msg = str(row[PROMPT_COL]) + PROMPT_SUFFIX
    assistant_msg = str(row[ANSWER_COL])
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds_train = Dataset.from_pandas(df).map(format_kienngx, num_proc=2, remove_columns=list(df.columns))
print(f"[OK] Formatted: {len(ds_train)}")
print(f"Sample (400 chars):")
print(ds_train[0]["text"][:400])


In [ ]:
# Cell 7: SFT kienngx - compatível com ambos paths Unsloth/transformers
from trl import SFTTrainer, SFTConfig
import threading, time, trl, torch
print(f"Using TRL {trl.__version__}")

args = SFTConfig(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    bf16=True,
    optim="adamw_torch",
    max_length=MAX_SEQ,
    dataset_text_field="text",
    packing=False,
    # Gradient checkpointing: True se transformers, False se Unsloth (ja ativa no Cell 5)
    gradient_checkpointing=not USED_UNSLOTH,
    gradient_checkpointing_kwargs={"use_reentrant": False} if not USED_UNSLOTH else None,
    max_grad_norm=1.0,
    report_to="none",
    push_to_hub=False,
    seed=42,
    remove_unused_columns=True,
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model, train_dataset=ds_train,
    args=args, processing_class=tok,
)

def monitor_mem():
    while True:
        try:
            m = torch.cuda.memory_allocated() / 1e9
            p = torch.cuda.max_memory_allocated() / 1e9
            print(f"[MEM] current={m:.1f}GB peak={p:.1f}GB")
        except: pass
        time.sleep(300)
threading.Thread(target=monitor_mem, daemon=True).start()

resume = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        resume = True
        print(f"Resuming ({len(ckpts)} ckpts)")

hours = 3 if USED_UNSLOTH else 5
print(f"Starting SFT ({hours}-{hours+2}h esperado)...")
stats = trainer.train(resume_from_checkpoint=resume)
print(f"Training done. Loss: {stats.training_loss:.4f}")


In [ ]:
# Cell 8: Save + validate gate + submission.zip + HF upload
import os, json, zipfile
from huggingface_hub import HfApi

FINAL_DIR = f"{CKPT_DIR}/final_adapter"
trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)
print(f"[OK] Saved: {FINAL_DIR}")
print(f"Files: {os.listdir(FINAL_DIR)}")

with open(f"{FINAL_DIR}/adapter_config.json") as f: cfg = json.load(f)
tm = cfg.get("target_modules", [])
rank = cfg.get("r", cfg.get("lora_rank", 0))
print(f"\n=== GATE VALIDATION ===")
print(f"target_modules: {tm}")
print(f"rank: {rank}")

errors = []
if not isinstance(tm, list) or not tm: errors.append("target_modules invalid")
else:
    if "in_proj" not in tm: errors.append("missing in_proj")
    if "gate_proj" in tm: errors.append("has gate_proj")
    if "x_proj" in tm: errors.append("has x_proj")
if rank > 32: errors.append(f"rank {rank} > 32")

if errors: print(f"!!! GATE FAIL: {errors}")
else: print("[OK] PASSES submission_gate local")

SUBMISSION_ZIP = f"{CKPT_DIR}/submission.zip"
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["adapter_config.json", "adapter_model.safetensors"]:
        src = os.path.join(FINAL_DIR, f)
        if os.path.exists(src):
            zf.write(src, arcname=f)
            print(f"  added: {f} ({os.path.getsize(src)/1e6:.1f} MB)")

api = HfApi(token=HF_TOKEN)
REPO = "felipesp1983/kg1-nemotron-lora-v73-robusto"
try:
    api.create_repo(REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=FINAL_DIR, repo_id=REPO, path_in_repo="final")
    api.upload_file(path_or_fileobj=SUBMISSION_ZIP, repo_id=REPO, path_in_repo="submission.zip")
    print(f"[OK] HF: https://huggingface.co/{REPO}")
except Exception as e:
    print(f"HF upload falhou: {e}")

print(f"\n" + "="*60)
print("KAGGLE SUBMIT:")
print("="*60)
print(f"1. Download: {SUBMISSION_ZIP}")
print(f"2. python scripts/local_score.py --adapter {REPO} --target-score 0.84")
print(f"3. python scripts/submit_kaggle.py --hf-repo {REPO} --message 'v73 robusto loss {stats.training_loss:.3f}'")
print(f"Score esperado: 0.85 +/- 0.02")


## Troubleshooting

### Cell 2 assert VRAM < 24GB
Precisa GPU >= 24GB. Runtime -> Change runtime type -> GPU T4/V100 nao serve.

### Cell 4 ambos falharem (Unsloth + transformers)
Esgotou possibilidades tecnicas. Opcoes:
- Tentar H100 80GB (comprovado V1)
- Runtime -> Disconnect -> reconectar (novo GPU)
- Mudar modelo: Nemotron-3-Nano-12B (cabe em qualquer GPU)

### Cell 4 `mamba-ssm required`
Patch automatico aplicado (fallback). Se persistir: `!pip install mamba-ssm` + Cell 4 retry.

### Cell 7 OOM durante treino
Peak > VRAM disponivel. Reduza max_length no Cell 7 para 1024 ou 1536.

### Cell 7 Loss > 10 step 10
Config errada. Possiveis causas:
- Gated rmsnorm patch incorreto (mamba-ssm not installed)
- target_modules nao matching

## Score projection
- P(score >= 0.86): ~40-50%
- P(score >= 0.85): ~70%
- P(score >= 0.84): ~85%
- Valor esperado: 0.85 +/- 0.02